# Notebook 02 — NumPy Transformations, Dimension Expansion, and Reshaping

**Session:** Array 1D Using NumPy  
**Module:** 02 · Calculus and Linear Algebra  
**Master Roadmap:** [Back to Roadmap](../../../README.md) · [Module 02 Index](../../README.md) · [Study Guide](../README.md)

---

## Learning Objectives

1. Restrict and bound numerical arrays using `np.clip` for gradient stabilization and signal bounding.
2. Concatenate arrays along specific axes using `np.concatenate` and diagnose dimension incompatibility errors.
3. Expand array dimensions and adjust tensor rank using `np.newaxis` and `np.expand_dims`.
4. Distinguish between deep copies (`flatten`) and memory views (`ravel`).
5. Reshape arrays into target dimensions and use `-1` for automatic dimension inference.


## 1. Value Clipping (`np.clip`)

The clipping operation constrains all elements of an array to lie within a designated numerical interval $[a, b]$:

$$\text{clip}(x, a, b) = \min(\max(x, a), b)$$

In Machine Learning, clipping is essential for:
- Preventing **exploding gradients** during backpropagation.
- Safeguarding log-loss computations: $\hat{y} \in [\epsilon, 1 - \epsilon]$ to prevent $\log(0)$.
- Clamping image pixel color intensities to $[0, 255]$ or $[0.0, 1.0]$.


In [ ]:
import numpy as np

# Exercise 1 from lecture slides
arr = np.array([1, 5, 8, 10, 3, 6])
clipped_arr = np.clip(arr, a_min=3, a_max=8)

print("Original array: ", arr)
print("Clipped [3, 8]: ", clipped_arr)

assert np.array_equal(clipped_arr, np.array([3, 5, 8, 8, 3, 6]))

# Machine Learning Application: Gradient Explosion Mitigation
simulated_gradients = np.array([0.02, 14.85, -23.10, 0.45, -0.01])
clipped_gradients = np.clip(simulated_gradients, a_min=-5.0, a_max=5.0)

print("\nSimulated Gradients: ", simulated_gradients)
print("Clipped Gradients:   ", clipped_gradients)
assert np.all(clipped_gradients >= -5.0) and np.all(clipped_gradients <= 5.0)
print("\nClipping assertions verified.")

## 2. Array Concatenation Along Axes (`np.concatenate`)

`np.concatenate((a1, a2, ...), axis=k)` joins a sequence of arrays along an existing axis. All input arrays must have identical shapes along every axis except the concatenation axis $k$.


In [ ]:
# 1. 1D Array Concatenation (axis=0)
arr1 = np.array([1, 2, 3])
arr2 = np.array([4, 5, 6])
cat_1d = np.concatenate((arr1, arr2))
print("1D Concatenation (6,):", cat_1d)
assert np.array_equal(cat_1d, np.array([1, 2, 3, 4, 5, 6]))

# 2. 2D Array Concatenation along axis=0 (vertical stacking of rows)
mat1 = np.array([[1, 2, 3]])
mat2 = np.array([[4, 5, 6]])
cat_2d_rows = np.concatenate((mat1, mat2), axis=0)
print("\n2D Concatenation along axis=0 (2x3):\n", cat_2d_rows)
assert cat_2d_rows.shape == (2, 3)

# 3. 2D Array Concatenation along axis=1 (horizontal stacking of columns)
cat_2d_cols = np.concatenate((mat1, mat2), axis=1)
print("\n2D Concatenation along axis=1 (1x6):\n", cat_2d_cols)
assert cat_2d_cols.shape == (1, 6)

# 4. Dimension Incompatibility Pitfall:
# Attempting to concatenate 1D arrays along axis=1 raises AxisError
try:
    invalid_cat = np.concatenate((arr1, arr2), axis=1)
except np.exceptions.AxisError as e:
    print("\nCaught expected AxisError for 1D concatenation along axis=1:")
    print(" ", e)

## 3. Expanding Dimensions with `np.newaxis`

`np.newaxis` (syntactic alias for `None`) inserts a new axis of length 1 at the specified position. This is crucial for:
- Converting 1D vectors `(N,)` into row vectors `(1, N)` or column vectors `(N, 1)`.
- Enabling broadcasting across matrix operations (e.g. computing outer products).


In [ ]:
vec = np.array([1, 2, 3, 4, 5])
print("Original 1D Vector shape:", vec.shape)

# Transform into row vector (1, 5)
row_vec = vec[np.newaxis, :]
print("Row Vector shape (1, 5):", row_vec.shape)

# Transform into column vector (5, 1)
col_vec = vec[:, np.newaxis]
print("Column Vector shape (5, 1):", col_vec.shape)

# Multi-axis expansion (5, 1, 1, 1)
tensor_vec = vec[:, np.newaxis, np.newaxis, np.newaxis]
print("Tensor Vector shape:", tensor_vec.shape)

# Application: Outer Product via Broadcasting
# x in (N, 1) * y in (1, M) -> Outer product (N, M)
x = np.array([1, 2, 3])[:, np.newaxis]
y = np.array([10, 20, 30, 40])[np.newaxis, :]
outer_prod = x * y

print("\nOuter Product Matrix (3x4):\n", outer_prod)
assert outer_prod.shape == (3, 4)
assert outer_prod[2, 3] == 3 * 40
print("\nDimension expansion assertions verified.")

## 4. Flattening Arrays: `flatten()` vs `ravel()`

- `arr.flatten()` returns a **deep copy** of the array flattened into 1D. Mutating it does not modify the source array.
- `arr.ravel()` returns a **memory view** whenever possible. Mutating it modifies the source array.


In [ ]:
# 1. arr.flatten() (Deep Copy)
orig_mat = np.array([[1, 2, 3], [4, 5, 6]])
flat_copy = orig_mat.flatten()
flat_copy[0] = 999

print("Source matrix after flatten mutation:\n", orig_mat)
print("Flattened copy:", flat_copy)
assert orig_mat[0, 0] == 1  # Unaffected by copy mutation

# 2. arr.ravel() (Memory View)
orig_mat2 = np.array([[1, 2, 3], [4, 5, 6]])
flat_view = orig_mat2.ravel()
flat_view[0] = 999

print("\nSource matrix after ravel mutation:\n", orig_mat2)
print("Ravel view:", flat_view)
assert orig_mat2[0, 0] == 999  # Modified through view!
print("\nFlatten vs Ravel memory semantics verified.")

## 5. Reshaping Arrays (`reshape`)

`arr.reshape(new_shape)` reinterprets the array dimensions without modifying data:
1. Total elements must be preserved: $\prod \text{new\_shape}_i = N$.
2. One dimension can be specified as `-1`, allowing NumPy to automatically infer its value:
$$\text{dim}_{\text{inferred}} = \frac{N}{\prod_{j \neq \text{inferred}} d_j}$$


In [ ]:
seq = np.array([1, 2, 3, 4, 5, 6])

# 1. Reshape into 2x3 matrix
mat_2x3 = seq.reshape((2, 3))
print("Reshaped (2, 3):\n", mat_2x3)

# 2. Reshape into 3x2 matrix
mat_3x2 = seq.reshape((3, 2))
print("\nReshaped (3, 2):\n", mat_3x2)

# 3. Reshape with automatic dimension inference (-1)
inferred_rows = seq.reshape((-1, 3))
inferred_cols = seq.reshape((2, -1))
print("\nInferred rows shape:", inferred_rows.shape)
print("Inferred cols shape:", inferred_cols.shape)
assert inferred_rows.shape == (2, 3)
assert inferred_cols.shape == (2, 3)

# 4. Multi-dimensional expansion
high_dim = seq.reshape((1, 1, 1, 2, 3))
print("\nHigh-dimensional tensor shape:", high_dim.shape)
assert high_dim.shape == (1, 1, 1, 2, 3)

# 5. Incompatible dimension error
try:
    bad_reshape = seq.reshape((4, 2))  # 4 * 2 = 8 != 6
except ValueError as e:
    print("\nCaught expected ValueError on incompatible shape:")
    print(" ", e)

## 6. Synthesis Challenge

1. Take a 1D vector of integers from 0 to 11.
2. Reshape it into a $3 \times 4$ matrix.
3. Extract column index 2 as a 1D slice.
4. Expand its dimension to a column vector $(3, 1)$ using `np.newaxis`.
5. Clip its values between 3 and 8.
6. Verify all shapes and values.


In [ ]:
challenge_vec = np.arange(12)
challenge_mat = challenge_vec.reshape(3, 4)
col_slice = challenge_mat[:, 2]
col_expanded = col_slice[:, np.newaxis]
col_clipped = np.clip(col_expanded, 3, 8)

print("Matrix (3x4):\n", challenge_mat)
print("Column 2 slice:", col_slice)
print("Expanded shape:", col_expanded.shape)
print("Clipped column vector:\n", col_clipped)

assert challenge_mat.shape == (3, 4)
assert np.array_equal(col_slice, np.array([2, 6, 10]))
assert np.array_equal(col_clipped, np.array([[3], [6], [8]]))
print("\nAll Notebook 02 validations passed successfully!")